In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import json
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from collections import Counter
import gc

DRIVE_PATH = '/content/drive/MyDrive/CICIoT2023_Research'

ATTACK_TAXONOMY = {
    'DDoS':       ['DDOS-ICMP_FLOOD','DDOS-UDP_FLOOD','DDOS-TCP_FLOOD','DDOS-PSHACK_FLOOD',
                   'DDOS-SYN_FLOOD','DDOS-RSTFINFLOOD','DDOS-SYNONYMOUSIP_FLOOD',
                   'DDOS-ACK_FRAGMENTATION','DDOS-UDP_FRAGMENTATION','DDOS-ICMP_FRAGMENTATION',
                   'DDOS-SLOWLORIS','DDOS-HTTP_FLOOD'],
    'DoS':        ['DOS-UDP_FLOOD','DOS-TCP_FLOOD','DOS-SYN_FLOOD','DOS-HTTP_FLOOD'],
    'Mirai':      ['MIRAI-GREETH_FLOOD','MIRAI-UDPPLAIN','MIRAI-GREIP_FLOOD'],
    'Recon':      ['RECON-HOSTDISCOVERY','RECON-OSSCAN','RECON-PORTSCAN','RECON-PINGSWEEP','VULNERABILITYSCAN'],
    'Spoofing':   ['MITM-ARPSPOOFING','DNS_SPOOFING'],
    'Web':        ['XSS','COMMANDINJECTION','BACKDOOR_MALWARE','SQLINJECTION','UPLOADING_ATTACK','BROWSERHIJACKING'],
    'BruteForce': ['DICTIONARYBRUTEFORCE'],
    'Benign':     ['BENIGN']
}

LABEL_TO_CATEGORY = {label: cat for cat, labels in ATTACK_TAXONOMY.items() for label in labels}
CATEGORY_ENCODING = {cat: i for i, cat in enumerate(ATTACK_TAXONOMY.keys())}
print(CATEGORY_ENCODING)

{'DDoS': 0, 'DoS': 1, 'Mirai': 2, 'Recon': 3, 'Spoofing': 4, 'Web': 5, 'BruteForce': 6, 'Benign': 7}


In [ ]:
import os
files = os.listdir(DRIVE_PATH)
print(f"Files in Drive folder: {files}")

Files in Drive folder: ['label_mapping.json', 'X_test.csv', 'X_train.csv', 'y_test_multi.csv', 'X_val.csv', 'y_train_multi.csv', 'y_val_multi.csv']


In [ ]:
with open(f'{DRIVE_PATH}/label_mapping.json') as f:
    label_mapping = json.load(f)

int_to_name = {v: k for k, v in label_mapping.items()}

print("Loading y_train...")
y_train_fine = pd.read_csv(
    f'{DRIVE_PATH}/y_train_multi.csv',
    header=0,        # treat first row as header
    dtype=int
).squeeze()

y_train_cat = y_train_fine.map(int_to_name).map(LABEL_TO_CATEGORY).map(CATEGORY_ENCODING)

print("Category distribution (train):")
print(y_train_cat.value_counts().sort_index())
print(f"\nNull count: {y_train_cat.isna().sum()}")

Loading y_train...
Category distribution (train):
target_multi
0    8605001
1    2924805
2    1651751
3     458610
4     305042
5      16522
6       8759
7     733115
Name: count, dtype: int64

Null count: 0


In [ ]:
print("Loading X_train...")
X_train = pd.read_csv(f'{DRIVE_PATH}/X_train.csv')
print(f"X_train shape: {X_train.shape}")
gc.collect()

Loading X_train...
X_train shape: (14703605, 39)


430

In [ ]:
UNDER_TARGETS = {
    CATEGORY_ENCODING['DDoS']:     500_000,
    CATEGORY_ENCODING['DoS']:      500_000,
    CATEGORY_ENCODING['Mirai']:    500_000,
    CATEGORY_ENCODING['Benign']:   500_000,
    CATEGORY_ENCODING['Recon']:    200_000,
    CATEGORY_ENCODING['Spoofing']: 200_000,
}

current_counts = Counter(y_train_cat)
sampling_strategy = {k: v for k, v in UNDER_TARGETS.items() if current_counts[k] > v}

print("Applying RandomUnderSampler...")
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_under, y_under = rus.fit_resample(X_train, y_train_cat)

print(f"After undersampling: {X_under.shape}")
print(Counter(y_under))

del X_train, y_train_cat, y_train_fine
gc.collect()

Applying RandomUnderSampler...
After undersampling: (2425281, 39)
Counter({0: 500000, 1: 500000, 2: 500000, 7: 500000, 3: 200000, 4: 200000, 5: 16522, 6: 8759})


0

In [ ]:
OVER_TARGETS = {
    CATEGORY_ENCODING['Web']:        100_000,
    CATEGORY_ENCODING['BruteForce']:  50_000,
}

print("Applying SMOTE-ENN...")
smote_enn = SMOTEENN(sampling_strategy=OVER_TARGETS, random_state=42, n_jobs=-1)
X_bal, y_bal = smote_enn.fit_resample(X_under, y_under)

print(f"After SMOTE-ENN: {X_bal.shape}")
print(Counter(y_bal))

del X_under, y_under
gc.collect()

Applying SMOTE-ENN...
After SMOTE-ENN: (1629654, 39)
Counter({2: 497762, 0: 310101, 7: 300680, 1: 247767, 4: 105434, 3: 72947, 5: 62880, 6: 32083})


0

In [ ]:
print("Saving to Drive...")

X_bal_df = pd.DataFrame(X_bal)
y_bal_ser = pd.Series(y_bal, name="category_label")

X_bal_df.to_csv(f'{DRIVE_PATH}/X_train_balanced_8cat_smoteenn.csv', index=False)
y_bal_ser.to_csv(f'{DRIVE_PATH}/y_train_balanced_8cat_smoteenn.csv', index=False)

with open(f'{DRIVE_PATH}/category_encoding_8cat.json', 'w') as f:
    json.dump(CATEGORY_ENCODING, f, indent=2)

from collections import Counter
final_counts = {str(k): int(v) for k, v in Counter(y_bal).items()}
with open(f'{DRIVE_PATH}/balanced_class_counts_8cat_smoteenn.json', 'w') as f:
    json.dump(final_counts, f, indent=2)

print("Done! Saved:")
print("  X_train_balanced_8cat_smoteenn.csv")
print("  y_train_balanced_8cat_smoteenn.csv")
print("  category_encoding_8cat.json")
print("  balanced_class_counts_8cat_smoteenn.json")
print("Final shape:", X_bal_df.shape, y_bal_ser.shape)

Saving to Drive...
Done! Saved:
  X_train_balanced_8cat_smoteenn.csv
  y_train_balanced_8cat_smoteenn.csv
  category_encoding_8cat.json
  balanced_class_counts_8cat_smoteenn.json
Final shape: (1629654, 39) (1629654,)
